In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import torch

print(f"PyTorch version: {torch.version.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    device = 0
else:
    device = -1
print(f"Using device: {device}")

PyTorch version: 2.6.0+cu124
CUDA available: True
CUDA device count: 1
GPU name: NVIDIA GeForce RTX 4050 Laptop GPU
GPU memory: 6.4 GB
Using device: 0


### Generate sentiment scores csv

In [2]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)
from torch.utils.data import DataLoader
import torch
import pandas as pd
from tqdm.auto import tqdm
import time

In [ ]:
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment"

CSV_PATH = "../../data/processed/ratings_clean.csv"

OUTPUT_FILE = "../../data/processed/ratings_clean.csv"

BATCH_SIZE = 512
MAX_LENGTH = 128


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

ds = load_dataset(
    "csv",
    data_files=CSV_PATH,
    split="train"
)

total_rows = len(ds)

print(f"Dataset loaded: {total_rows:,} rows")


texts = [
    (t[:500] if t else "")
    for t in ds["text"]
]


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME
)

model.to(device)
model.eval()


loader = DataLoader(
    texts,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

label_scores = torch.tensor(
    [-1.0, 0.0, 1.0],
    device=device
)

results = []

start_time = time.time()

progress_bar = tqdm(
    loader,
    total=len(loader),
    desc="Sentiment Inference",
    unit="batch",
    dynamic_ncols=True
)

with torch.no_grad():

    for batch_texts in progress_bar:

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
)

        encoded = {
            k: v.to(device, non_blocking=True)
            for k, v in encoded.items()
        }

        outputs = model(**encoded)

        logits = outputs.logits

        probs = torch.softmax(logits, dim=-1)

        signed_scores = (probs * label_scores).sum(dim=1)

        results.extend(
            signed_scores.cpu().tolist()
        )

        processed = len(results)

        elapsed = time.time() - start_time

        speed = processed / elapsed

        remaining = total_rows - processed

        eta_minutes = (remaining / speed) / 60

        progress_bar.set_postfix({
            "speed": f"{speed:.1f}/sec",
            "eta": f"{eta_minutes:.1f}m",
            "done": f"{processed:,}"
        })

elapsed_total = time.time() - start_time

print("\nInference complete")
print(f"Total time: {elapsed_total / 60:.2f} minutes")
print(f"Average speed: {total_rows / elapsed_total:.2f} reviews/sec")

df = ds.to_pandas()

df["sentiment_score"] = results

df.to_csv(OUTPUT_FILE, index=False)

print(f"Saved to: {OUTPUT_FILE}")

In [ ]:
# create a file with item_id and avg sentiment score for each item_id
df = pd.read_csv("../../data/processed/ratings_clean.csv")

sentiment_by_item = df.groupby("item_id")["sentiment_score"].mean().reset_index()
sentiment_by_item.to_csv("../../data/processed/item_sentiment_scores.csv", index=False)

In [5]:
# set global avg setiment score for whole dataset
df = pd.read_csv("../../data/processed/ratings_clean.csv")
global_avg_sentiment = df["sentiment_score"].mean()
print(f"Global average sentiment score: {global_avg_sentiment:.4f}")

Global average sentiment score: 0.5481


In [12]:
# Smooth item sentiment scores by blending the item average with the global average
item_sentiment = pd.read_csv("../../data/processed/item_sentiment_scores.csv")
review_counts = df.groupby("item_id").size().rename("review_count").reset_index()
C = int(review_counts["review_count"].median())

item_scores = item_sentiment.merge(review_counts, on="item_id", how="left")
item_scores["smoothed_sentiment_score"] = (
    item_scores["review_count"] * item_scores["sentiment_score"]
    + C * global_avg_sentiment
    ) / (item_scores["review_count"] + C)

item_scores[["item_id", "sentiment_score", "smoothed_sentiment_score"]].to_csv(
    "../../data/processed/item_sentiment_scores.csv",
    index=False
 )

In [16]:
import numpy as np

low = np.percentile(item_scores["smoothed_sentiment_score"], 33)
high = np.percentile(item_scores["smoothed_sentiment_score"], 66)

print(f"Low threshold: {low:.4f}")
print(f"High threshold: {high:.4f}")

def assign_label(score):
    if score >= high:
        return "Highly Praised"
    elif score >= low:
        return "Mixed Reviews"
    else:
        return "Niche Appeal"
    
item_scores["label"] = item_scores["smoothed_sentiment_score"].apply(assign_label)
item_scores.to_csv(
    "../../data/processed/item_sentiment_scores.csv",
    index=False
)

Low threshold: 0.4946
High threshold: 0.6069
